In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Optional, List
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

In [13]:
load_dotenv(override=True)

True

In [14]:
"""
ACE Review Workflow — Orchestrator-Workers Pattern
====================================================
Orchestrator  : reads N student submissions, dispatches N reviewer workers via Send()
Worker        : one reviewer per student → produces core_strengths, general_feedback, areas_to_improve
Aggregator    : collects all N structured reviews into final output
"""

import operator
from typing import Annotated, Optional
from typing_extensions import TypedDict


from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from pydantic import BaseModel, Field


# ──────────────────────────────────────────────
# LLM
# ──────────────────────────────────────────────

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


# ──────────────────────────────────────────────
# Structured output schema  (what reviewer returns)
# ──────────────────────────────────────────────

class ACEReview(BaseModel):
    """Structured evaluation output for one student."""
    general_feedback: str = Field(
        description="Overall feedback on the student's submission covering all provided sections."
    )
    core_strengths: str = Field(
        description="2-3 specific strengths demonstrated by the student based on their submission."
    )
    areas_to_improve: str = Field(
        description="2-3 concrete, actionable areas where the student should focus growth."
    )


reviewer_llm = llm.with_structured_output(ACEReview)


# ──────────────────────────────────────────────
# State definitions
# ──────────────────────────────────────────────

class StudentSubmission(TypedDict):
    """Input schema for a single student's review submission."""
    student_id: str
    student_name: str
    projects_worked_on: str            # mandatory
    certifications_achieved: Optional[str]  # optional
    extra_work_if_any: Optional[str]        # optional


class CompletedReview(TypedDict):
    """Output schema — one reviewer's evaluation of one student."""
    student_id: str
    student_name: str
    general_feedback: str
    core_strengths: str
    areas_to_improve: str


class OverallState(TypedDict):
    """
    Top-level graph state.
    `completed_reviews` uses operator.add as reducer so each
    parallel worker can safely append its result without collision.
    """
    students: list[StudentSubmission]
    completed_reviews: Annotated[list[CompletedReview], operator.add]


class WorkerState(TypedDict):
    """
    Scoped state passed to each individual reviewer worker.
    Each worker sees only its own student — keeps context clean.
    """
    student: StudentSubmission
    completed_reviews: Annotated[list[CompletedReview], operator.add]


# ──────────────────────────────────────────────
# Nodes
# ──────────────────────────────────────────────

def orchestrator(state: OverallState) -> list[Send]:
    """
    Reads the student list and dynamically spawns one reviewer
    worker per student using LangGraph's Send() API.
    Number of workers == number of students (determined at runtime).
    """
    return [
        Send("reviewer_worker", {"student": student, "completed_reviews": []})
        for student in state["students"]
    ]


def reviewer_worker(state: WorkerState) -> dict:
    """
    Reviewer agent for a single student.
    - Always evaluates: projects_worked_on
    - Conditionally includes: certifications_achieved, extra_work_if_any
    Produces structured ACEReview output.
    """
    student = state["student"]

    # ── Build the review content dynamically (optional fields handled here) ──
    review_sections = [
        f"**Projects worked on:**\n{student['projects_worked_on']}"
    ]

    if student.get("certifications_achieved"):
        review_sections.append(
            f"**Certifications achieved:**\n{student['certifications_achieved']}"
        )

    if student.get("extra_work_if_any"):
        review_sections.append(
            f"**Extra work / initiatives:**\n{student['extra_work_if_any']}"
        )

    review_content = "\n\n".join(review_sections)

    # ── Prompt ──
    system_prompt = """You are an expert academic reviewer evaluating a student's performance submission.

Your job is to produce a fair, specific, and constructive evaluation with exactly three components:
1. general_feedback  — holistic assessment of everything the student submitted
2. core_strengths    — 2-3 specific strengths backed by evidence from their submission
3. areas_to_improve  — 2-3 actionable, concrete improvement areas (not vague)

Be encouraging but honest. Base your evaluation strictly on what is provided."""

    human_prompt = f"""Please evaluate the following student submission:

Student: {student['student_name']} (ID: {student['student_id']})

{review_content}

Note: Only certifications and extra work sections present were provided; evaluate accordingly."""

    # ── LLM call with structured output ──
    result: ACEReview = reviewer_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt)
    ])

    # ── Return as CompletedReview, appended via operator.add reducer ──
    completed = CompletedReview(
        student_id=student["student_id"],
        student_name=student["student_name"],
        general_feedback=result.general_feedback,
        core_strengths=result.core_strengths,
        areas_to_improve=result.areas_to_improve,
    )

    return {"completed_reviews": [completed]}


# ──────────────────────────────────────────────
# Graph construction
# ──────────────────────────────────────────────

def build_ace_review_graph() -> StateGraph:
    builder = StateGraph(OverallState)

    # Register nodes
    builder.add_node("orchestrator", orchestrator)
    builder.add_node("reviewer_worker", reviewer_worker)

    # Edges
    # START → orchestrator (via conditional edge using Send API for dynamic fan-out)
    builder.add_conditional_edges(
        START,
        orchestrator,
        ["reviewer_worker"]   # tells LangGraph which nodes Send() can target
    )

    # Each reviewer worker → END (fan-in is automatic via operator.add reducer)
    builder.add_edge("reviewer_worker", END)

    return builder.compile()


# ──────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────

if __name__ == "__main__":

    graph = build_ace_review_graph()

    # ── Sample input: mix of complete and partial submissions ──
    sample_students: list[StudentSubmission] = [
        {
            "student_id": "S001",
            "student_name": "Priya Sharma",
            "projects_worked_on": "Built a RAG chatbot using LangChain and OpenSearch. Led backend API development for a client dashboard in FastAPI.",
            "certifications_achieved": "AWS Certified Solutions Architect – Associate, Azure AI Fundamentals",
            "extra_work_if_any": "Conducted an internal LLM workshop for 15 junior developers. Presented at the Gurgaon AI Engineers Meetup.",
        },
        {
            "student_id": "S002",
            "student_name": "Rohan Mehta",
            "projects_worked_on": "Developed a data pipeline for e-commerce sales forecasting using PySpark. Contributed to a recommender system POC.",
            "certifications_achieved": None,          # did not submit certifications
            "extra_work_if_any": "Mentored 2 interns during Q3.",
        },
        {
            "student_id": "S003",
            "student_name": "Anjali Verma",
            "projects_worked_on": "Implemented a document classification model using fine-tuned BERT. Integrated model into a Flask microservice.",
            "certifications_achieved": "Google Professional Data Engineer",
            "extra_work_if_any": None,                # no extra activities submitted
        },
    ]

    print("=" * 60)
    print("ACE Review Workflow — Starting")
    print(f"Students to evaluate: {len(sample_students)}")
    print("=" * 60)

    result = graph.invoke({"students": sample_students, "completed_reviews": []})

    # ── Print results ──
    for review in result["completed_reviews"]:
        print(f"\n{'─' * 50}")
        print(f"Student : {review['student_name']} ({review['student_id']})")
        print(f"{'─' * 50}")
        print(f"\n📋 General Feedback:\n{review['general_feedback']}")
        print(f"\n💪 Core Strengths:\n{review['core_strengths']}")
        print(f"\n🎯 Areas to Improve:\n{review['areas_to_improve']}")

    print(f"\n{'=' * 60}")
    print(f"Done. {len(result['completed_reviews'])} reviews completed.")
    print("=" * 60)

ACE Review Workflow — Starting
Students to evaluate: 3

──────────────────────────────────────────────────
Student : Priya Sharma (S001)
──────────────────────────────────────────────────

📋 General Feedback:
Priya has demonstrated a strong commitment to professional development, as evidenced by her achievement of notable certifications such as the AWS Certified Solutions Architect – Associate and Azure AI Fundamentals. Additionally, her initiative in conducting an internal LLM workshop and presenting at the Gurgaon AI Engineers Meetup showcases her expertise and willingness to share knowledge with others.

💪 Core Strengths:
Priya's strengths include her ability to achieve prestigious certifications, her willingness to take on leadership roles in sharing knowledge, and her ability to work on a wide range of projects, from chatbot development to backend API development.

🎯 Areas to Improve:
To further grow, Priya could focus on documenting her project experiences in more detail, explori